In [1]:
%pip install -q open_clip_torch transformers timm huggingface_hub fastapi pyngrok uvicorn nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00


In [ ]:
import torch
import open_clip
from fastapi import FastAPI, UploadFile, File, Form
import nest_asyncio
from PIL import Image
import io
from typing import Any
import json
import torch.nn.functional as F

nest_asyncio.apply()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_NAME = (
    "hf-hub:"
    "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)

model, preprocess_traion, preprocess_val = open_clip.create_model_and_transforms(MODEL_NAME)

candidate_texts = [
    "Normal shoulder MRI",
    "Rotator cuff tear",
    "Shoulder joint effusion",
    "Labral tear",
    "Bone marrow edema",
    "impingement syndrome"
]

tokenizer = open_clip.get_tokenizer(MODEL_NAME)
text_tokens = tokenizer(candidate_texts)

print("text token shape:", text_tokens.shape)
print("text token dtype:", text_tokens.dtype)
print("text token device:", text_tokens.device)

text_tokens = text_tokens.to(device)

print("text token device:", text_tokens.device)

model = model.to(device)
model.eval()

print("모델 로드 완료")
print("device:", device)
print("training mode:", model.training)

app = FastAPI()

@app.post("/predict")
async def predict(image_info:  str = Form(...), images: list[UploadFile] = File(...)):
    parsed_image_info: list[dict[str, Any]] = json.loads(image_info)
    image_items = []
    for file in images:
        filename = file.filename.split('_')[0]
        slice_number = file.filename.split('_')[1].replace(".png", "")
        meta = next(file_data for file_data in parsed_image_info if file_data.get("description") == filename)
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        # print("received image size:", image.size)
        # print("received image mode:", image.mode)

        image_tensor = preprocess_val(image)
        image_items.append({
            "description": meta.get("description"),
            "slice_number": slice_number,
            "tensor": preprocess_val(image)
        })

    image_batch = torch.stack([x["tensor"] for x in image_items]).to(device)

    # print("preprocessed tensor shape:", image_batch.shape)
    # print("preprocessed tensor dtype:", image_tensor.dtype)
    # print("preprocessed tensor device:", image_tensor.device)

    with torch.inference_mode():
        image_features = model.encode_image(image_batch)
        text_features = model.encode_text(text_tokens)

        print("text feature shape:", text_features.shape)
        print("text feature dtype:", text_features.dtype)
        print("text feature device:", text_features.device)

        print("image feature shape:", image_features.shape)

        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        print("normalized image feature shape:", image_features.shape)
        print("image feature norm:", image_features.norm(dim=-1))
        print("text feature norms:", text_features.norm(dim=-1))

        similarity = image_features @ text_features.T
        print("similarity shape:", similarity.shape)
        print("similarity:", similarity)
        # probabilities = F.softmax(similarity, dim=1)

    probabilities = torch.softmax(similarity, dim=1)
    predicted_indices = probabilities.argmax(dim=1)
    # predicted_scores = probabilities.max(dim=1).values

    for item, label_indices, probs in zip(image_items, predicted_indices, probabilities):
        meta = next(file_data for file_data in parsed_image_info if file_data.get("description") == item.get("description"))
        if int(label_indices.item()) == 0:
            continue
            
        meta["result"].append({
            "slice_number": item.get("slice_number"),
            "text": candidate_texts[int(label_indices.item())],
            "score": [{"text": text, "score": f"{score * 100:.2f}%"} for text, score in zip(candidate_texts, probs)]
        })

    print(f"parsed_image_info: {parsed_image_info}")
    return parsed_image_info

        # best_index = similarity.argmax(dim=-1).item()
        # best_text = candidate_texts[best_index]
        # best_score = similarity[0, best_index].item()

        # print("best index:", best_index)
        # print("best prediction:", best_text)
        # print("best score:", best_score)

        # total_score = {}
        # for text, score in zip(candidate_texts, similarity[0]):
        #     total_score[text] = score.item()

    # return {
    #     "filename": file.filename,
    #     "best_index": best_index,
    #     "best_text": best_text,
    #     "best_score": best_score,
    #     "total_score": total_score
    # }



from pyngrok import ngrok
import asyncio
import uvicorn

NGROK_TOKEN = "3EzUV00bhdXtnBDjZ6hMUxdpIsv_7sYr1oc1Kf28T1JJrg6Yz"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000, domain="overrate-comprised-outfield.ngrok-free.dev")
print(f"\n 외부 프로세스에서 접근 가능한 API 주소-vlm: {public_url.public_url}/predict\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="debug")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.create_task(server.serve())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


text token shape: torch.Size([6, 256])
text token dtype: torch.int64
text token device: cpu
text token device: cuda:0
모델 로드 완료
device: cuda
training mode: False

 외부 프로세스에서 접근 가능한 API 주소-vlm: https://overrate-comprised-outfield.ngrok-free.dev/predict



<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:76>>

INFO:     Started server process [6029]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


text feature shape: torch.Size([6, 512])
text feature dtype: torch.float32
text feature device: cuda:0
image feature shape: torch.Size([183, 512])
normalized image feature shape: torch.Size([183, 512])
image feature norm: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.

In [ ]:
CANDIDATES = [
    {
        "id": "normal",
        "prompt": (
            "a normal coronal T2 fat-suppressed "
            "shoulder MRI"
        ),
    },
    {
        "id": "rotator_cuff_tendinosis",
        "prompt": (
            "a coronal T2 fat-suppressed shoulder MRI "
            "showing rotator cuff tendinosis"
        ),
    },
    {
        "id": "partial_thickness_rotator_cuff_tear",
        "prompt": (
            "a coronal T2 fat-suppressed shoulder MRI "
            "showing a partial-thickness rotator cuff tear"
        ),
    },
    {
        "id": "full_thickness_rotator_cuff_tear",
        "prompt": (
            "a coronal T2 fat-suppressed shoulder MRI "
            "showing a full-thickness rotator cuff tear"
        ),
    },
    {
        "id": "subacromial_subdeltoid_bursitis",
        "prompt": (
            "a coronal T2 fat-suppressed shoulder MRI "
            "showing subacromial-subdeltoid bursitis"
        ),
    },
]

for candidate in CANDIDATES:
    print(candidate["id"])
    print(candidate["prompt"])
    print()

candidate_ids = [
    candidate["id"]
    for candidate in CANDIDATES
                    ]

candidate_prompts = [
    candidate["prompt"]
    for candidate in CANDIDATES
]

text_tokens = tokenizer(candidate_prompts).to(device)
with torch.inference_mode():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1,keepdim=True)

print(text_features.shape)

print(text_features[0])